# 01 — Data Setup & Dataset Audit

**Project**: Spam Email Detection  
**Phase**: 1 — Project Setup & Dataset Audit  
**Dataset**: UCI SMS Spam Collection  

| Cell section | Member attribution |
|---|---|
| Environment & config import | Member 2 (Config Lead) |
| Data ingestion & rename | Member 1 (Data Lead) |
| Shape / dtype / null audit | Member 1 (Data Lead) |
| Duplicate detection & removal | Member 1 (Data Lead) |
| Label distribution | Member 3 (Documentation Lead) |
| Label encoding | Member 2 (Config Lead) |
| Save processed output | Member 1 (Data Lead) |

> **Note**: This notebook performs QC only. No vectorisation, no `train_test_split` execution, no text normalisation — those belong to later phases.

---
## Cell 1 — Environment & Config Import
**Member 2 (Config Lead)**: Import all project-wide constants from `config.py`. 
No magic numbers appear anywhere below — every constant references `config.*`.

In [ ]:
# Member 2 (Config Lead): Import standard libraries and project config
import sys
import os

import pandas as pd
import numpy as np

# ── Ensure project root is on path so config.py is importable ─────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import config  # noqa: E402

print("=" * 60)
print("PROJECT CONFIGURATION")
print("=" * 60)
print(f"  RAW_DATA_PATH      : {config.RAW_DATA_PATH}")
print(f"  PROCESSED_DATA_PATH: {config.PROCESSED_DATA_PATH}")
print(f"  RANDOM_STATE       : {config.RANDOM_STATE}")
print(f"  TEST_SIZE          : {config.TEST_SIZE}  (configured; NOT executed here)")
print(f"  LABEL_MAP          : {config.LABEL_MAP}")
print(f"  LABEL_COL          : {config.LABEL_COL}")
print(f"  LABEL_NUM_COL      : {config.LABEL_NUM_COL}")
print(f"  TEXT_COL           : {config.TEXT_COL}")
print("=" * 60)

---
## Cell 2 — Data Ingestion
**Member 1 (Data Lead)**: Load the raw CSV using `encoding='latin-1'` (required for this dataset — it contains extended ASCII characters that UTF-8 cannot parse). Drop the three unnamed artefact columns produced by the original CSV export. Rename `v1 → label`, `v2 → message`.

In [ ]:
# Member 1 (Data Lead): Load raw dataset

RAW_PATH = os.path.join(PROJECT_ROOT, config.RAW_DATA_PATH)

# ── Guard: fail fast with a clear message if the file is missing ──────────────
if not os.path.exists(RAW_PATH):
    raise FileNotFoundError(
        f"\n\nRaw dataset not found at: {RAW_PATH}\n"
        "Please follow the manual download steps in data/README.md\n"
        "and place spam.csv at data/raw/spam.csv before running this notebook."
    )

# ── Load with latin-1 encoding; keep all columns first to inspect ─────────────
df_raw = pd.read_csv(
    RAW_PATH,
    encoding="latin-1",
)

print(f"Raw shape (before any cleaning): {df_raw.shape}")
print("\nRaw columns:", df_raw.columns.tolist())
print("\nFirst 3 rows (raw):")
df_raw.head(3)

In [ ]:
# Member 1 (Data Lead): Drop unnamed artefact columns and rename v1/v2

# Identify and drop all 'Unnamed' columns produced by CSV export artefact
unnamed_cols = [c for c in df_raw.columns if c.startswith("Unnamed")]
print(f"Dropping {len(unnamed_cols)} unnamed column(s): {unnamed_cols}")

df = df_raw.drop(columns=unnamed_cols).copy()

# Rename v1 -> label, v2 -> message (using constants from config)
df.rename(
    columns={"v1": config.LABEL_COL, "v2": config.TEXT_COL},
    inplace=True
)

print(f"\nColumns after rename: {df.columns.tolist()}")
print(f"Shape after column cleanup: {df.shape}")
print("\nFirst 5 rows:")
df.head()

---
## Cell 3 — Shape, Dtypes & Missing Value Audit
**Member 1 (Data Lead)**: Produce a comprehensive quality report — dimensions, column types, and a per-column null count. Any nulls in `label` or `message` would be a critical data integrity failure.

In [ ]:
# Member 1 (Data Lead): Shape, dtype, and null audit

print("=" * 60)
print("DATASET QUALITY REPORT — POST COLUMN CLEANUP")
print("=" * 60)

print(f"\n  Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\n  Column dtypes:")
for col, dtype in df.dtypes.items():
    print(f"    {col:<15} {str(dtype):<10}")

print("\n  Missing value counts per column:")
nulls = df.isnull().sum()
for col, count in nulls.items():
    flag = "  ⚠ NULLS DETECTED" if count > 0 else "  ✓ OK"
    print(f"    {col:<15} {count:>6} null(s){flag}")

total_nulls = nulls.sum()
print(f"\n  Total nulls in dataset: {total_nulls}")

if total_nulls == 0:
    print("  ✅ No missing values — dataset is complete.")
else:
    print("  ❌ Missing values detected — investigate before proceeding.")

---
## Cell 4 — Exact Duplicate Detection & Removal
**Member 1 (Data Lead)**: Identify fully identical rows. Duplicates can inflate model accuracy metrics and skew class-distribution counts. Log before/after row counts for the audit trail.

In [ ]:
# Member 1 (Data Lead): Exact-duplicate detection and removal

rows_before = len(df)
n_duplicates = df.duplicated().sum()

print("=" * 60)
print("DUPLICATE AUDIT")
print("=" * 60)
print(f"  Rows BEFORE dedup : {rows_before:,}")
print(f"  Exact duplicates   : {n_duplicates:,}")

if n_duplicates > 0:
    print("\n  Sample duplicate rows (first 5):")
    dupe_mask = df.duplicated(keep=False)
    display(df[dupe_mask].head(10))
    
    # Drop duplicates — keep first occurrence
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    print(f"\n  → Duplicates removed. Keeping first occurrence.")
else:
    print("  → No exact duplicates found.")

rows_after = len(df)
print(f"\n  Rows AFTER dedup  : {rows_after:,}")
print(f"  Rows removed       : {rows_before - rows_after:,}")
print("=" * 60)

---
## Cell 5 — Label Distribution
**Member 3 (Documentation Lead)**: Compute the class balance. The dataset is known to be imbalanced (~87% ham / ~13% spam) — this is documented but NOT corrected here (Phase 3 concern).

In [ ]:
# Member 3 (Documentation Lead): Label distribution analysis

print("=" * 60)
print("LABEL DISTRIBUTION")
print("=" * 60)

label_counts = df[config.LABEL_COL].value_counts()
label_pct    = df[config.LABEL_COL].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({
    "Count": label_counts,
    "Percentage (%)": label_pct.round(2)
})
dist_df.index.name = "Class"
print(dist_df.to_string())

total = label_counts.sum()
print(f"\n  Total messages (post-dedup): {total:,}")

imbalance_ratio = label_counts.max() / label_counts.min()
print(f"  Imbalance ratio (majority:minority): {imbalance_ratio:.1f}:1")
print("\n  ⚠ Class imbalance noted — to be addressed in Phase 3 (modelling).")
print("  ✅ No resampling or correction applied in this phase.")
print("=" * 60)

In [ ]:
# Member 3 (Documentation Lead): Raw text length statistics (logged only — NOT used for filtering)
# Phase 2 EDA will produce full visualisations; this is a quick sanity check.

df["_msg_len_chars"] = df[config.TEXT_COL].str.len()
df["_msg_len_words"] = df[config.TEXT_COL].str.split().str.len()

print("RAW TEXT LENGTH STATS (logged; text is NOT modified here)")
print("=" * 60)
print("  Character length by class:")
print(df.groupby(config.LABEL_COL)["_msg_len_chars"].describe().round(1).to_string())
print("\n  Word count by class:")
print(df.groupby(config.LABEL_COL)["_msg_len_words"].describe().round(1).to_string())

# Drop temp columns — not saved to processed file
df.drop(columns=["_msg_len_chars", "_msg_len_words"], inplace=True)
print("\n  Temp stat columns dropped — clean dataframe retained.")

---
## Cell 6 — Label Encoding
**Member 2 (Config Lead)**: Map `label` (string) → `label_num` (int) using `LABEL_MAP` from `config.py`. Both columns are preserved in the output — the string label for human readability, the numeric label for sklearn compatibility in Phase 3.

In [ ]:
# Member 2 (Config Lead): Apply LABEL_MAP from config to create numeric label column

df[config.LABEL_NUM_COL] = df[config.LABEL_COL].map(config.LABEL_MAP)

print("=" * 60)
print("LABEL ENCODING")
print("=" * 60)
print(f"  Mapping applied  : {config.LABEL_MAP}")
print(f"  Source column    : '{config.LABEL_COL}' (string, unchanged)")
print(f"  New column       : '{config.LABEL_NUM_COL}' (int)")

# Verify no unmapped labels produced NaN
unmapped = df[config.LABEL_NUM_COL].isnull().sum()
print(f"\n  Unmapped / NaN in '{config.LABEL_NUM_COL}': {unmapped}")
if unmapped == 0:
    print("  ✅ All labels successfully encoded.")
else:
    raise ValueError(f"{unmapped} rows have unmapped labels — check LABEL_MAP in config.py")

# Confirm unique value pairs
print("\n  Unique (label, label_num) pairs:")
print(df[[config.LABEL_COL, config.LABEL_NUM_COL]].drop_duplicates().to_string(index=False))

print("\n  Final column order:")
print(f"  {df.columns.tolist()}")
print("\n  Sample rows:")
display(df.sample(5, random_state=config.RANDOM_STATE))

---
## Cell 7 — Final QC Summary & Save
**Member 1 (Data Lead)**: Run a final integrity check, then write the cleaned dataframe to `data/processed/clean_data.csv`. The output path is taken from `config.PROCESSED_DATA_PATH`.

In [ ]:
# Member 1 (Data Lead): Final integrity assertions before saving

print("=" * 60)
print("FINAL QC ASSERTIONS")
print("=" * 60)

# 1. Zero nulls
final_nulls = df.isnull().sum().sum()
assert final_nulls == 0, f"FAIL: {final_nulls} null values remain!"
print(f"  [PASS] Zero null values in cleaned dataframe ({final_nulls} nulls).")

# 2. Zero exact duplicates
final_dupes = df.duplicated().sum()
assert final_dupes == 0, f"FAIL: {final_dupes} duplicate rows remain!"
print(f"  [PASS] Zero exact duplicates ({final_dupes} dupes).")

# 3. Expected columns present
expected_cols = {config.LABEL_COL, config.TEXT_COL, config.LABEL_NUM_COL}
assert expected_cols.issubset(set(df.columns)), f"FAIL: Missing columns {expected_cols - set(df.columns)}"
print(f"  [PASS] All expected columns present: {sorted(expected_cols)}.")

# 4. label_num contains only 0 and 1
valid_labels = set(config.LABEL_MAP.values())
actual_labels = set(df[config.LABEL_NUM_COL].unique())
assert actual_labels == valid_labels, f"FAIL: Unexpected label_num values: {actual_labels}"
print(f"  [PASS] label_num contains only valid values: {sorted(valid_labels)}.")

print(f"\n  Final dataframe shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\n  ✅ All QC checks passed. Ready to save.")
print("=" * 60)

In [ ]:
# Member 1 (Data Lead): Save cleaned dataframe to data/processed/clean_data.csv

PROCESSED_PATH = os.path.join(PROJECT_ROOT, config.PROCESSED_DATA_PATH)

# Ensure the processed directory exists
os.makedirs(os.path.dirname(PROCESSED_PATH), exist_ok=True)

# Save — index=False so pandas doesn't write a spurious row index column
df.to_csv(PROCESSED_PATH, index=False, encoding="utf-8")

# Verify the saved file
df_verify = pd.read_csv(PROCESSED_PATH)

print("=" * 60)
print("SAVE VERIFICATION")
print("=" * 60)
print(f"  Saved to     : {PROCESSED_PATH}")
print(f"  File size    : {os.path.getsize(PROCESSED_PATH) / 1024:.1f} KB")
print(f"  Loaded shape : {df_verify.shape[0]:,} rows × {df_verify.shape[1]} columns")
print(f"  Columns      : {df_verify.columns.tolist()}")
print(f"  Nulls in saved file: {df_verify.isnull().sum().sum()}")
print()
print("  ✅ Phase 1 complete. clean_data.csv is ready for Phase 2 EDA.")
print("=" * 60)
df_verify.head()

---
## Phase 1 Summary

| Step | Action | Result |
|------|--------|--------|
| Load raw CSV | `encoding='latin-1'`, all columns | ✅ |
| Drop unnamed cols | 3 artefact columns removed | ✅ |
| Rename columns | `v1→label`, `v2→message` | ✅ |
| Null audit | 0 nulls in label + message | ✅ |
| Duplicate removal | Exact dupes detected & dropped | ✅ |
| Label distribution | Logged; imbalance noted, not corrected | ✅ |
| Raw text stats | Char/word lengths logged by class | ✅ |
| Label encoding | `label_num` column added via LABEL_MAP | ✅ |
| Final QC | 4 assertions: nulls, dupes, cols, labels | ✅ |
| Save output | `data/processed/clean_data.csv` | ✅ |

**Next**: Phase 2 — `02_eda_and_preprocessing.ipynb`  
EDA visualisations, text normalisation (lowercasing, stop-words, stemming), and feature analysis.